# ClearML Dataset 教学 Notebook（全场景）

教材来源：`CLEARML_DATASET_GUIDE.md`

学习目标（按教材顺序）：

1. 先记住最小工作流：`create -> add_files/add_external_files -> upload -> finalize`
2. 再用 3 个问题串起来：创建上传 / 训练消费 / 增量维护
3. 最后按场景套模板（本 notebook 就是这些场景的可运行版）


## 使用说明

- 建议从上到下依次运行所有代码单元
- 运行前请确认你已经执行过 `clearml-init`，并能连接到 ClearML Server

本 notebook 只关注 Dataset 的核心流程，不额外引入存储配置等内容。


## 先记住最小工作流（最重要）

生产一个数据集版本（数据生产者）：

```python
from clearml import Dataset

ds = Dataset.create(dataset_project="demo", dataset_name="demo_dataset")
ds.add_files("/path/to/data")
ds.upload()
ds.finalize()
```

训练/推理消费一个数据集（数据消费者）：

```python
from clearml import Dataset

dataset_path = Dataset.get(dataset_project="demo", dataset_name="demo_dataset").get_local_copy()
```


## 三个问题（你学 Dataset 就围绕它们）

1) 我怎么创建并上传一个数据集版本？

- 关键 API：`Dataset.create / add_files / add_external_files / upload / finalize`

2) 我怎么在训练/推理代码里使用一个数据集？

- 关键 API：`Dataset.get / get_local_copy / alias / overridable`

3) 我怎么基于旧版本做增量更新？

- 关键 API：`create(parent_datasets=...) / sync_folder / remove_files / upload / finalize`


## 场景选型速查（只保留最核心的 5 个）

| 场景 | 推荐方法 |
|---|---|
| 本地目录注册成数据集 | `create + add_files + upload + finalize` |
| 训练代码拿到数据路径 | `get + get_local_copy` |
| 任务里记录数据集依赖 | `Task.init + get(alias=...)` |
| 基于旧版本增量更新 | `create(parent_datasets=[...])` |
| 本地继续加工一份副本 | `get_mutable_local_copy` |


In [ ]:
# 配置单元：只保留教学必要的 3 个变量

from datetime import datetime
from pathlib import Path

from clearml import Dataset, Task

# ClearML 项目名
PROJECT_NAME = 'clearml_dataset_demo'

# 数据集名字（加时间戳避免和历史运行冲突）
DATASET_NAME = f"toy_dataset_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# 本地生成玩具数据的目录
WORKSPACE = Path('/mnt/md0/zhouyunqi/clearml/demo_artifacts/clearml_dataset_demo_notebook').resolve()

print('PROJECT_NAME =', PROJECT_NAME)
print('DATASET_NAME =', DATASET_NAME)
print('WORKSPACE    =', WORKSPACE)


## 场景 0：准备玩具数据（非 Dataset）

目的：提供可一键运行的最小数据，不让你卡在“准备数据”这一步。

关键点：这部分与 Dataset 无关，所以代码会尽量简化并封装到小函数里。

你将在本地看到：`WORKSPACE/base_source`、`WORKSPACE/sync_source`、`WORKSPACE/external_store`。


In [ ]:
# 非 Dataset 相关操作：用很短的辅助函数生成玩具数据
# 这里的函数内部细节不用关心，你只需要知道它会产出 base_source / sync_source / external_store。

import csv
import json
import shutil
from pathlib import Path
from typing import Dict, Iterable

def _write_csv(path: Path, rows: Iterable[Dict[str, object]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

def _write_json(path: Path, payload: Dict[str, object]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')

def _reset_dir(path: Path) -> None:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

def build_demo_files(workspace: Path) -> Dict[str, Path]:
    _reset_dir(workspace)

    base_source = workspace / 'base_source'
    sync_source = workspace / 'sync_source'
    external_store = workspace / 'external_store'

    _write_csv(
        base_source / 'tabular' / 'train.csv',
        [
            {'sample_id': 1, 'feature': 0.10, 'label': 'low'},
            {'sample_id': 2, 'feature': 0.45, 'label': 'mid'},
            {'sample_id': 3, 'feature': 0.90, 'label': 'high'},
        ],
    )
    _write_json(
        base_source / 'meta' / 'labels.json',
        {'low': 0, 'mid': 1, 'high': 2},
    )
    (base_source / 'docs').mkdir(parents=True, exist_ok=True)
    (base_source / 'docs' / 'notes.txt').write_text('v1 才有，v2 会删除\n', encoding='utf-8')

    _write_csv(
        sync_source / 'train.csv',
        [
            {'sample_id': 1, 'feature': 0.11, 'label': 'low'},
            {'sample_id': 2, 'feature': 0.44, 'label': 'mid'},
            {'sample_id': 3, 'feature': 0.92, 'label': 'high'},
        ],
    )
    _write_csv(
        sync_source / 'valid.csv',
        [
            {'sample_id': 101, 'feature': 0.20, 'label': 'low'},
            {'sample_id': 102, 'feature': 0.60, 'label': 'mid'},
        ],
    )

    _write_csv(
        external_store / 'reference.csv',
        [
            {'bucket': 'A', 'threshold': 0.30},
            {'bucket': 'B', 'threshold': 0.70},
        ],
    )

    return {
        'workspace': workspace,
        'base_source': base_source,
        'sync_source': sync_source,
        'external_store': external_store,
    }


In [ ]:
# 生成玩具数据

paths = build_demo_files(WORKSPACE)

print("已生成演示目录：")
for k, v in paths.items():
    print(f"  {k}: {v}")

print("\nbase_source 文件：")
for p in sorted(paths["base_source"].rglob("*")):
    if p.is_file():
        print("  -", p.relative_to(paths["base_source"]))


## 场景 1：本地目录创建 Dataset（v1 基础版本）

目的：用“最小工作流”创建第一个可复用版本（v1）。

关键 API：`Dataset.create`、`add_files`、`upload`、`finalize`。

你将在 UI 看到：

- Datasets 页面出现一个同名数据集版本（v1）
- 文件树里能看到 `tabular/`、`meta/`、`docs/`

常见坑：忘记 `finalize()` 会导致后续复用/继承/下载不稳定。


In [ ]:
# v1：本地目录创建 Dataset

ds_v1 = Dataset.create(
    dataset_project=PROJECT_NAME,
    dataset_name=DATASET_NAME,
    description='v1：本地文件创建的基础版本',
)

# 把本地目录登记到数据集内部路径（建议显式写 dataset_path）
ds_v1.add_files(paths['base_source'] / 'tabular', dataset_path='tabular')
ds_v1.add_files(paths['base_source'] / 'meta', dataset_path='meta')
ds_v1.add_files(paths['base_source'] / 'docs', dataset_path='docs')

# 上传 + 冻结（最小工作流的关键两步）
ds_v1.upload()
ds_v1.finalize()

print('v1 dataset_id =', ds_v1.id)


## 场景 2：父子版本增量更新（v2 子版本）

目的：基于 v1 派生一个新版本 v2，并演示“增量维护”的常用手法。

关键 API：

- 继承：`Dataset.create(parent_datasets=[ds_v1.id])`
- 同步：`sync_folder(local_path=..., dataset_path=...)`
- 外部文件：`add_external_files(source_url=...)`
- 删除：`remove_files(...)`
- 收尾：`upload`、`finalize`

你将在 UI 看到：

- v2 版本显示 parent = v1 的血缘关系
- `docs/notes.txt` 不再出现在 v2
- `external/` 路径下多出外部文件条目

常见坑：`sync_folder` 只负责登记差异，别忘了后续 `upload()` + `finalize()`。


In [ ]:
# v2：基于 v1 派生子版本（增量维护）

ds_v2 = Dataset.create(
    dataset_project=PROJECT_NAME,
    dataset_name=DATASET_NAME,
    parent_datasets=[ds_v1.id],
    description='v2：继承 v1，演示 sync_folder / add_external_files / remove_files',
)

# 1) 同步目录到 tabular/（本地目录是“当前真实状态”时很常用）
ds_v2.sync_folder(
    local_path=paths['sync_source'],
    dataset_path='tabular',
    verbose=False,
)

# 2) 登记外部文件（这里用 file:// 演示；真实项目常用 s3://）
ds_v2.add_external_files(
    source_url=(paths['external_store'] / 'reference.csv').as_uri(),
    dataset_path='external',
)

# 3) 删除 v1 中不再需要的文件
ds_v2.remove_files('docs/notes.txt')

# 上传 + 冻结
ds_v2.upload()
ds_v2.finalize()

print('v2 dataset_id =', ds_v2.id)


## 场景 3：训练/推理任务中消费数据集（记录依赖）

目的：在训练/推理任务中“拿到可复现的数据路径”，并把数据集依赖写进 Task。

关键 API：

- `Task.init`（让你在 UI 里看到这次消费行为）
- `Dataset.get(alias=..., overridable=True)`（记录依赖、便于远端替换）
- `get_local_copy()`（拿只读本地路径，训练直接用）

你将在 UI 看到：

- 项目里出现一个 `*_consumer_demo` 的 Task
- Task 里能看到 alias 对应的数据集引用（便于复现）

常见坑：必须在 `Task.init()` 之后调用带 `alias` 的 `Dataset.get()`，否则不会绑定到当前 Task。


In [ ]:
# 场景 3：训练/推理任务中消费 Dataset（记录依赖）

from pathlib import Path

task = Task.init(
    project_name=PROJECT_NAME,
    task_name=f"{DATASET_NAME}_consumer_demo",
    task_type=Task.TaskTypes.data_processing,
)

# alias 会把“这次任务用的是哪个数据集”记录到 Task 里
dataset_by_id = Dataset.get(
    dataset_id=ds_v2.id,
    alias='train_data',
)

local_copy = Path(dataset_by_id.get_local_copy())
print('local_copy =', local_copy)

task.close()


## 场景 4：需要本地可写目录时（可写副本）

目的：当你需要“下载后继续加工/写新文件”时，使用可写副本而不是只读缓存。

关键 API：`get_mutable_local_copy(target_folder=..., overwrite=True)`。

你将在本地看到：`WORKSPACE/mutable_local_copy` 下生成一份可写目录副本。

常见坑：训练消费请优先用 `get_local_copy()`；只有确实要写文件才用 `get_mutable_local_copy()`。


In [ ]:
# 获取可写副本

writable_target = WORKSPACE / "mutable_local_copy"
if writable_target.exists():
    shutil.rmtree(writable_target)

writable_path = dataset_by_id.get_mutable_local_copy(
    target_folder=writable_target,
    overwrite=True,
)

print("writable_path =", writable_path)


## 场景 5：快速验证（最小可见结果）

目的：让你确认下载路径正确、文件结构正确。

检查点：

- `tabular/train.csv`
- `meta/labels.json`


In [ ]:
train_csv = local_copy / "tabular" / "train.csv"
labels_json = local_copy / "meta" / "labels.json"

print("train.csv 内容：")
print(train_csv.read_text(encoding="utf-8"))

print("labels.json 内容：")
print(labels_json.read_text(encoding="utf-8"))


## 运行后你应该在 ClearML UI 里看到什么

1) 在项目 `PROJECT_NAME` 下：

- Dataset：同名 Dataset 的两个版本（v1 和 v2），并且 v2 显示血缘关系（parent = v1）
- 你能在 Dataset 页面看到文件结构：`tabular/`、`meta/`、`external/` 等

2) 同一个项目下还会出现一个 Task：

- `*_consumer_demo`
- 它会记录 `alias=train_data` 对应的数据集引用（方便复现与远端替换）
